# Machine Learning: Prediksi Status Pasien Sirosis Hati

Alur: **dataset → EDA → pre-processing → pemodelan → evaluasi → review/ringkasan**

Cara pakai di Google Colab: jalankan sel dari atas ke bawah (Shift + Enter).
Saat sel upload muncul, pilih file **ML.xlsx**.

| Tahap | Isi |
|---|---|
| 1 | Import library dan load data |
| 2 | Data understanding dan perbaikan kualitas data |
| 3 | EDA (univariat, bivariat, korelasi, missing value, outlier) |
| 4 | Pre-processing (split, imputasi, transformasi, encoding, scaling) |
| 5 | Pemodelan dan cross-validation |
| 6 | Hyperparameter tuning |
| 7 | Evaluasi di test set |
| 8 | Interpretasi model |
| 9 | Ringkasan untuk review |

## Pengaturan (bisa diubah)

In [ ]:
TARGET = "Status"          # C = censored (hidup), CL = censored karena transplantasi, D = meninggal
GABUNG_C_CL = False        # True -> jadi 2 kelas: D vs bukan D (berguna karena CL hanya ~25 data)
PAKAI_NDAYS = False        # N_Days = lama follow-up; sangat terkait hasil akhir (rawan leakage). Coba True untuk pembanding.
RANDOM_STATE = 42
TEST_SIZE = 0.2

## 1. Import library dan load data

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, accuracy_score, f1_score, balanced_accuracy_score)
from sklearn.inspection import permutation_importance

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

In [ ]:
# Upload file di Colab. Kalau dijalankan di komputer lokal, otomatis pakai file ML.xlsx di folder yang sama.
try:
    from google.colab import files
    up = files.upload()
    PATH = list(up.keys())[0]
except ImportError:
    PATH = "ML.xlsx"

df_raw = pd.read_excel(PATH)
df = df_raw.copy()
print("Ukuran data:", df.shape)
df.head()

## 2. Data understanding

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

### 2.1 Data dictionary (ringkas)

| Kolom | Arti | Tipe |
|---|---|---|
| ID | nomor pasien | identitas (dibuang) |
| N_Days | jumlah hari antara registrasi dan kejadian akhir (meninggal/transplantasi/akhir studi) | numerik |
| **Status** | status pasien: C, CL, D | **target** |
| Drug | D-penicillamine atau Placebo | kategorik |
| Age | umur dalam **hari** | numerik |
| Sex | F / M | kategorik |
| Ascites, Hepatomegaly, Spiders | ada/tidak (Y/N) | kategorik |
| Edema | N, S (ada tanpa/dengan diuretik), Y | kategorik |
| Bilirubin, Cholesterol, Albumin, Copper, Alk_Phos, SGOT, Tryglicerides, Platelets, Prothrombin | hasil lab | numerik |
| Stage | stadium histologis penyakit (1 sampai 4) | ordinal |

### 2.2 Perbaikan kualitas data: tanda desimal hilang

Pada file ini beberapa kolom lab tampak kehilangan tanda desimal saat diekspor
(contoh: Albumin 414 seharusnya 4,14; Prothrombin 122 seharusnya 12,2).
Kita cek lalu perbaiki memakai **rentang nilai klinis yang wajar**:
nilai dibagi 10 berulang kali sampai masuk batas atas yang masuk akal.

In [ ]:
cek = ["Bilirubin", "Albumin", "Prothrombin", "SGOT", "Alk_Phos"]
print("SEBELUM diperbaiki:")
display(df[cek].describe().T[["min", "50%", "max"]])

In [ ]:
def perbaiki_skala(s, batas_atas):
    """Bagi 10 berulang sampai nilai <= batas_atas (untuk nilai yang kehilangan desimal)."""
    s = s.astype(float).copy()
    for _ in range(4):
        m = s > batas_atas
        if not m.any():
            break
        s[m] = s[m] / 10
    return s

# Batas atas berdasarkan rentang klinis yang lazim pada penyakit hati:
batas = {"Albumin": 5.0, "Prothrombin": 20.0, "SGOT": 500.0, "Alk_Phos": 14000.0}
for kolom, b in batas.items():
    df[kolom] = perbaiki_skala(df[kolom], b)

print("SESUDAH diperbaiki:")
display(df[list(batas)].describe().T[["min", "50%", "max"]])

**Catatan penting (baca):**
- Perbaikan di atas adalah *perkiraan*. Nilai yang aslinya berakhiran ",0" (misal 1718,0) dan yang tidak
  kehilangan desimal tidak bisa dibedakan dengan pasti dari nilai yang sudah bergeser.
- **Bilirubin** tampaknya juga bergeser 10x (14,5 menjadi 145), tetapi nilai kecil seperti "14" bisa berarti 1,4 atau 14,0,
  sehingga **tidak bisa diperbaiki otomatis** dan dibiarkan apa adanya. Untuk hasil paling akurat, unduh ulang dataset asli
  (Kaggle: *Cirrhosis Prediction Dataset*, Mayo Clinic PBC) dan ganti file ML.xlsx.
- Tulis hal ini di bagian keterbatasan laporan Anda.

In [ ]:
# Feature engineering sederhana + buang kolom identitas
df["Age_years"] = df["Age"] / 365.25
df = df.drop(columns=["ID", "Age"])

# Target
if GABUNG_C_CL:
    df[TARGET] = df[TARGET].replace({"CL": "C"})
print(df[TARGET].value_counts())
print("\nProporsi (%):")
print((df[TARGET].value_counts(normalize=True) * 100).round(1))

## 3. EDA (Exploratory Data Analysis)

### 3.1 Distribusi target

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
sns.countplot(data=df, x=TARGET, order=df[TARGET].value_counts().index, ax=ax)
ax.set_title("Distribusi target (Status)")
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2, p.get_height()), ha="center", va="bottom")
plt.show()

### 3.2 Univariat: fitur numerik

In [ ]:
kat_cols = ["Drug", "Sex", "Ascites", "Hepatomegaly", "Spiders", "Edema", "Stage"]
num_cols = ["N_Days", "Age_years", "Bilirubin", "Cholesterol", "Albumin", "Copper",
            "Alk_Phos", "SGOT", "Tryglicerides", "Platelets", "Prothrombin"]

fig, axes = plt.subplots(3, 4, figsize=(16, 9))
for ax, c in zip(axes.ravel(), num_cols):
    sns.histplot(df[c].dropna(), kde=True, ax=ax)
    ax.set_title(f"{c} (skew={df[c].skew():.2f})")
for ax in axes.ravel()[len(num_cols):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

Fitur dengan *skewness* tinggi (misalnya > 1) akan ditransformasi log pada tahap pre-processing.

### 3.3 Univariat: fitur kategorik

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, c in zip(axes.ravel(), kat_cols):
    sns.countplot(data=df, x=c, ax=ax)
    ax.set_title(c)
for ax in axes.ravel()[len(kat_cols):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

### 3.4 Bivariat: fitur numerik vs Status

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, c in zip(axes.ravel(), num_cols):
    sns.boxplot(data=df, x=TARGET, y=c, ax=ax)
    ax.set_title(f"{c} vs {TARGET}")
for ax in axes.ravel()[len(num_cols):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Rata-rata dan median tiap fitur per kelas
df.groupby(TARGET)[num_cols].median().round(2).T

### 3.5 Bivariat: fitur kategorik vs Status

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, c in zip(axes.ravel(), kat_cols):
    ct = pd.crosstab(df[c].fillna("Missing"), df[TARGET], normalize="index") * 100
    ct.plot(kind="bar", stacked=True, ax=ax, legend=(c == kat_cols[0]))
    ax.set_title(f"{c} vs {TARGET} (%)"); ax.set_xlabel("")
for ax in axes.ravel()[len(kat_cols):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

### 3.6 Korelasi antar fitur numerik (Spearman)

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df[num_cols].corr(method="spearman"), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Korelasi Spearman"); plt.show()

### 3.7 Analisis missing value

In [ ]:
mv = pd.DataFrame({"jumlah_missing": df.isna().sum(),
                   "persen": (df.isna().mean() * 100).round(1)})
mv = mv[mv.jumlah_missing > 0].sort_values("persen", ascending=False)
display(mv)
print("Baris dengan minimal 1 missing:", df.isna().any(axis=1).sum(), "dari", len(df))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(df.sort_values("Drug", na_position="last").isna().T, cbar=False, ax=ax, yticklabels=True)
ax.set_title("Peta missing value (putih = ada, gelap = kosong). Baris diurutkan berdasarkan Drug.")
ax.set_xlabel("Pasien"); plt.show()

In [ ]:
# Apakah missing berkaitan dengan pasien yang tidak ikut uji klinis (Drug kosong)?
kol_lab = ["Ascites", "Hepatomegaly", "Spiders", "Cholesterol", "Copper", "Alk_Phos", "SGOT", "Tryglicerides"]
tanpa_drug = df["Drug"].isna()
tabel = pd.DataFrame({
    "% kosong jika Drug kosong": df.loc[tanpa_drug, kol_lab].isna().mean() * 100,
    "% kosong jika Drug ada":   df.loc[~tanpa_drug, kol_lab].isna().mean() * 100,
}).round(1)
tabel

**Interpretasi:** jika kolom-kolom di atas hampir selalu kosong ketika `Drug` kosong, berarti missing **tidak acak**
(pasien di luar uji klinis memang tidak diperiksa lengkap). Karena itu kita membuat kolom indikator missing
dan kategori "Unknown" agar informasi ini tidak hilang.

### 3.8 Deteksi outlier (IQR)

In [ ]:
def hitung_outlier(s):
    q1, q3 = s.quantile(.25), s.quantile(.75)
    iqr = q3 - q1
    return ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()

out = pd.DataFrame({"outlier_IQR": [hitung_outlier(df[c].dropna()) for c in num_cols],
                    "skewness": [df[c].skew() for c in num_cols]}, index=num_cols).round(2)
out.sort_values("skewness", ascending=False)

Pada data medis, nilai ekstrem sering valid (pasien berat), jadi outlier **tidak dibuang**.
Kita redam pengaruhnya dengan transformasi `log1p` pada fitur yang sangat miring.

## 4. Pre-processing

Urutan penting agar tidak terjadi *data leakage*: **split dulu**, baru imputasi/scaling (semua di dalam `Pipeline`
sehingga hanya "belajar" dari data train).

In [ ]:
fitur_dipakai_num = [c for c in num_cols if PAKAI_NDAYS or c != "N_Days"]
fitur_kat = ["Drug", "Sex", "Ascites", "Hepatomegaly", "Spiders", "Edema"]
fitur_ordinal = ["Stage"]

X = df[fitur_dipakai_num + fitur_kat + fitur_ordinal].copy()
le = LabelEncoder()
y = le.fit_transform(df[TARGET])
print("Kelas:", dict(zip(le.classes_, le.transform(le.classes_))))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
print("Train:", X_train.shape, "| Test:", X_test.shape)

In [ ]:
# Fitur yang sangat miring -> log1p
skew_cols = [c for c in fitur_dipakai_num if X_train[c].skew() > 1]
lurus_cols = [c for c in fitur_dipakai_num if c not in skew_cols]
print("Ditransformasi log1p:", skew_cols)
print("Tanpa log:", lurus_cols)

def buat_preprocessor():
    pipe_skew = Pipeline([
        ("imp", SimpleImputer(strategy="median", add_indicator=True)),
        ("log", FunctionTransformer(np.log1p, validate=False, feature_names_out="one-to-one")),
        ("sc", StandardScaler()),
    ])
    pipe_lurus = Pipeline([
        ("imp", SimpleImputer(strategy="median", add_indicator=True)),
        ("sc", StandardScaler()),
    ])
    pipe_kat = Pipeline([
        ("imp", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("ohe", OneHotEncoder(handle_unknown="ignore")),
    ])
    pipe_ord = Pipeline([
        ("imp", SimpleImputer(strategy="median", add_indicator=True)),
        ("sc", StandardScaler()),
    ])
    return ColumnTransformer([
        ("skew", pipe_skew, skew_cols),
        ("lurus", pipe_lurus, lurus_cols),
        ("kat", pipe_kat, fitur_kat),
        ("ord", pipe_ord, fitur_ordinal),
    ])

prep_demo = buat_preprocessor().fit(X_train)
print("Jumlah fitur setelah pre-processing:", prep_demo.transform(X_train).shape[1])

Ringkasan pre-processing yang dipakai:
1. Buang `ID`, ubah `Age` (hari) menjadi tahun, perbaiki skala kolom lab.
2. Split train/test 80:20 dengan `stratify`.
3. Imputasi: median (numerik) + kolom indikator missing; "Unknown" (kategorik).
4. `log1p` untuk fitur dengan skewness > 1.
5. One-Hot Encoding untuk kategorik; StandardScaler untuk numerik.
6. Imbalance ditangani dengan `class_weight="balanced"` pada model yang mendukung.

## 5. Pemodelan dan cross-validation

In [ ]:
modelspec = {
    "Logistic Regression": LogisticRegression(max_iter=3000, class_weight="balanced"),
    "Decision Tree":       DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest":       RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE),
    "Gradient Boosting":   HistGradientBoostingClassifier(class_weight="balanced", random_state=RANDOM_STATE),
    "SVM (RBF)":           SVC(probability=True, class_weight="balanced", random_state=RANDOM_STATE),
    "KNN":                 KNeighborsClassifier(n_neighbors=7),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"accuracy": "accuracy", "balanced_acc": "balanced_accuracy", "f1_macro": "f1_macro"}

hasil_cv = []
for nama, model in modelspec.items():
    pipe = Pipeline([("prep", buat_preprocessor()), ("clf", model)])
    s = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)
    hasil_cv.append({"Model": nama,
                     "Accuracy": s["test_accuracy"].mean(),
                     "Balanced Acc": s["test_balanced_acc"].mean(),
                     "F1 macro": s["test_f1_macro"].mean(),
                     "F1 macro (std)": s["test_f1_macro"].std()})

tabel_cv = pd.DataFrame(hasil_cv).set_index("Model").sort_values("F1 macro", ascending=False).round(3)
tabel_cv

In [ ]:
tabel_cv[["Accuracy", "Balanced Acc", "F1 macro"]].plot(kind="barh", figsize=(8, 4))
plt.title("Perbandingan model (5-fold CV di data train)"); plt.xlim(0, 1); plt.show()

## 6. Hyperparameter tuning

In [ ]:
param_space = {
    "Random Forest": {
        "clf__n_estimators": [200, 300, 500],
        "clf__max_depth": [None, 4, 6, 8, 12],
        "clf__min_samples_leaf": [1, 2, 4, 8],
        "clf__max_features": ["sqrt", 0.5],
    },
    "Gradient Boosting": {
        "clf__learning_rate": [0.02, 0.05, 0.1],
        "clf__max_depth": [2, 3, 4, None],
        "clf__max_leaf_nodes": [8, 15, 31],
        "clf__l2_regularization": [0.0, 0.1, 1.0],
        "clf__min_samples_leaf": [10, 20, 30],
    },
    "Logistic Regression": {"clf__C": [0.01, 0.1, 1, 10, 100]},
    "SVM (RBF)": {"clf__C": [0.1, 1, 10, 100], "clf__gamma": ["scale", 0.01, 0.1]},
    "Decision Tree": {"clf__max_depth": [3, 4, 5, 6, 8], "clf__min_samples_leaf": [1, 5, 10, 20]},
    "KNN": {"clf__n_neighbors": [3, 5, 7, 9, 15], "clf__weights": ["uniform", "distance"]},
}

model_terbaik = tabel_cv.index[0]
print("Model terbaik dari CV:", model_terbaik)

pipe_tuning = Pipeline([("prep", buat_preprocessor()), ("clf", modelspec[model_terbaik])])
search = RandomizedSearchCV(pipe_tuning, param_space[model_terbaik], n_iter=15, cv=cv,
                            scoring="f1_macro", random_state=RANDOM_STATE, n_jobs=-1)
search.fit(X_train, y_train)

print("Parameter terbaik:", search.best_params_)
print(f"F1 macro CV sebelum tuning: {tabel_cv.loc[model_terbaik, 'F1 macro']:.3f} | sesudah: {search.best_score_:.3f}")
final_model = search.best_estimator_

## 7. Evaluasi di test set

In [ ]:
y_pred = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)

print("Accuracy          :", round(accuracy_score(y_test, y_pred), 3))
print("Balanced accuracy :", round(balanced_accuracy_score(y_test, y_pred), 3))
print("F1 macro          :", round(f1_score(y_test, y_pred, average="macro"), 3))
print("F1 weighted       :", round(f1_score(y_test, y_pred, average="weighted"), 3))
if len(le.classes_) > 2:
    auc = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")
else:
    auc = roc_auc_score(y_test, y_proba[:, 1])
print("ROC-AUC (macro)   :", round(auc, 3))
print()
print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=le.classes_, ax=axes[0], cmap="Blues")
axes[0].set_title("Confusion matrix (jumlah)")
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=le.classes_, ax=axes[1],
                                        cmap="Blues", normalize="true", values_format=".2f")
axes[1].set_title("Confusion matrix (dinormalisasi per kelas asli)")
plt.tight_layout(); plt.show()

In [ ]:
# Cek overfitting: skor train vs test
f1_train = f1_score(y_train, final_model.predict(X_train), average="macro")
f1_test = f1_score(y_test, y_pred, average="macro")
print(f"F1 macro train: {f1_train:.3f} | test: {f1_test:.3f} | selisih: {f1_train - f1_test:.3f}")
print("Selisih besar (mis. > 0.15) menandakan indikasi overfitting.")

## 8. Interpretasi: fitur apa yang paling berpengaruh?

In [ ]:
pi = permutation_importance(final_model, X_test, y_test, scoring="f1_macro",
                            n_repeats=20, random_state=RANDOM_STATE, n_jobs=-1)
imp = pd.DataFrame({"fitur": X_test.columns, "penting": pi.importances_mean, "std": pi.importances_std})
imp = imp.sort_values("penting", ascending=True)

plt.figure(figsize=(7, 6))
plt.barh(imp["fitur"], imp["penting"], xerr=imp["std"])
plt.title("Permutation importance (penurunan F1 macro jika fitur diacak)")
plt.tight_layout(); plt.show()
imp.sort_values("penting", ascending=False).head(10).round(4)

### Error analysis
Perhatikan kelas mana yang paling sering salah prediksi pada confusion matrix. Biasanya kelas minoritas (**CL**) sulit
diprediksi karena datanya hanya sekitar 25 baris. Coba `GABUNG_C_CL = True` di bagian pengaturan dan bandingkan hasilnya.

## 9. Ringkasan untuk review

In [ ]:
ringkasan_prep = pd.DataFrame([
    ["Tanda desimal hilang (Albumin, Prothrombin, SGOT, Alk_Phos)", "Nilai dibagi 10 sampai masuk rentang klinis wajar"],
    ["Bilirubin diduga bergeser 10x", "Tidak diperbaiki otomatis (ambigu); dicatat sebagai keterbatasan"],
    ["Age dalam hari", "Diubah menjadi tahun (Age_years)"],
    ["Kolom ID", "Dibuang (bukan fitur)"],
    ["Missing value (hingga ~33% per kolom)", "Median + indikator missing (numerik); kategori 'Unknown' (kategorik)"],
    ["Distribusi miring / outlier", "Transformasi log1p (skewness > 1); outlier tidak dibuang"],
    ["Fitur kategorik", "One-Hot Encoding"],
    ["Skala fitur berbeda", "StandardScaler"],
    ["Kelas tidak seimbang", "class_weight='balanced' + metrik F1 macro / balanced accuracy"],
    ["Risiko data leakage", "Split dulu, semua transformasi di dalam Pipeline (fit hanya di train)"],
], columns=["Masalah pada data", "Penanganan"])
display(ringkasan_prep)

In [ ]:
print("=== RINGKASAN METODE ===")
print("Model yang dibandingkan :", ", ".join(modelspec.keys()))
print("Validasi                :", "Stratified 5-fold CV pada data train + test set 20% terpisah")
print("Model terpilih          :", model_terbaik)
print("Parameter terbaik       :", search.best_params_)
print()
print("=== HASIL DI TEST SET ===")
print(f"Accuracy {accuracy_score(y_test, y_pred):.3f} | Balanced Acc {balanced_accuracy_score(y_test, y_pred):.3f} | "
      f"F1 macro {f1_score(y_test, y_pred, average='macro'):.3f} | ROC-AUC {auc:.3f}")
print("Fitur paling berpengaruh:", ", ".join(imp.sort_values("penting", ascending=False)["fitur"].head(5)))
display(tabel_cv)

### Template kesimpulan (isi dengan angka hasil Anda)
- **Dataset:** 418 pasien, 19 fitur, target Status (C/CL/D), banyak missing value pada pasien di luar uji klinis.
- **Pre-processing:** perbaikan skala, imputasi median + indikator, log1p, one-hot, scaling, class weight.
- **Model terbaik:** ______ dengan F1 macro ______ dan ROC-AUC ______ di test set.
- **Insight klinis:** fitur paling berpengaruh ______ (misalnya bilirubin, albumin, prothrombin, umur).
- **Keterbatasan:** data kecil, kelas CL sangat sedikit, kolom lab kehilangan desimal, hasil belum divalidasi eksternal.
- **Saran:** unduh dataset asli, coba survival analysis (Cox), tambah data, atau gabungkan C dan CL.